# Touch and sight

Figures 2-4 and Section 5.

In [ ]:
import sys
sys.path.insert(0, "..")

from embodiment import config as cfg

import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

cfg.set_style()

books = pd.read_csv(cfg.BOOK_COUNTS)
allow = set(cfg.ENGLISH_ORIGINALS.read_text().split()) - {""}
books = books[books["filename"].isin(allow)].drop_duplicates(subset="filename")

for label in cfg.TOUCH_SIGHT:
    books[f"rate_{label}"] = (books[f"n_{label}"] / books["total_sentences"]
                              * 1000)
print(f"{len(books):,} works")

## Pooled shares

In [ ]:
total = books["total_sentences"].sum()
counts = {l: books[f"n_{l}"].sum() for l in cfg.TOUCH_SIGHT}

touch = counts["haptic_human_animal"] + counts["haptic_material"]
sight = counts["sight_human"] + counts["sight_material"]
print(f"touch: {counts['haptic_material'] / touch:.0%} material-directed")
print(f"sight: {counts['sight_human'] / sight:.0%} person-directed")
for label, n in counts.items():
    print(f"  {label:22s} {n:>9,}  ({n / total * 100:.2f}% of sentences)")

## Figure 2

In [ ]:
COLOURS = [cfg.BLUE, cfg.AQUA, cfg.BLUE, cfg.AQUA]
positions = [0.79, 1.21, 1.79, 2.21]
data = [books[f"rate_{l}"] for l in cfg.TOUCH_SIGHT]

fig, ax = plt.subplots(figsize=(8, 5.5))
bp = ax.boxplot(data, positions=positions, widths=0.36, patch_artist=True,
                showfliers=False, zorder=3,
                medianprops=dict(color=cfg.INK, linewidth=2),
                whiskerprops=dict(color=cfg.MUTED, linewidth=1.5),
                capprops=dict(color=cfg.MUTED, linewidth=1.5))
for patch, colour in zip(bp["boxes"], COLOURS):
    patch.set_facecolor(colour)
    patch.set_edgecolor(cfg.SURFACE)
    patch.set_linewidth(2)

# Median above the median line, offset in points so the gap is the same in
# every box whatever the y-scale.
for pos, series in zip(positions, data):
    ax.annotate(f"{series.median():.1f}", xy=(pos, series.median()),
                xytext=(0, 4), textcoords="offset points", ha="center",
                va="bottom", fontsize=9.5, color=cfg.INK)

ax.set_xticks([1, 2])
ax.set_xticklabels(["Touch", "Sight"], fontsize=12, color=cfg.INK)
ax.set_ylabel("Per-book rate (per 1,000 sentences)")
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.yaxis.grid(True, color=cfg.GRID, linewidth=1, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(left=False, bottom=False)
ax.margins(y=0.06)
ax.legend(handles=[mpatches.Patch(color=cfg.BLUE,
                                  label="Human / animal-directed"),
                   mpatches.Patch(color=cfg.AQUA,
                                  label="Material / environment-directed")],
          frameon=False, loc="upper right", fontsize=10)
fig.tight_layout()
cfg.save_fig(fig, "fig2_touch_sight_boxplot")

for label, series in zip(cfg.TOUCH_SIGHT, data):
    q1, med, q3 = series.quantile([0.25, 0.5, 0.75])
    print(f"  {label:22s} median {med:5.1f}   IQR {q1:5.1f}-{q3:5.1f}")

## Figure 3

In [ ]:
# Each book's own rate first, then the mean across books in the decade, so a
# long novel does not outweigh a short one. Pre-1810 decades hold <10 books.
decade = (books[books["decade"] >= 1810].groupby("decade")
          .agg(n_books=("filename", "count"),
               **{f"{l}_mean": (f"rate_{l}", "mean") for l in cfg.TOUCH_SIGHT},
               **{f"{l}_sem": (f"rate_{l}", "sem") for l in cfg.TOUCH_SIGHT})
          .reset_index())

PANELS = [("Touch", ["haptic_human_animal", "haptic_material"]),
          ("Sight", ["sight_human", "sight_material"])]
SERIES = {"haptic_human_animal": "Human / animal-directed",
          "haptic_material": "Material / environment-directed",
          "sight_human": "Human / animal-directed",
          "sight_material": "Material / environment-directed"}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, (panel, pair), letter in zip(axes, PANELS, "ab"):
    for label, colour in zip(pair, (cfg.BLUE, cfg.AQUA)):
        mean, sem = decade[f"{label}_mean"], decade[f"{label}_sem"]
        ax.plot(decade["decade"], mean, color=colour, linewidth=2, marker="o",
                markersize=5.5, label=SERIES[label], zorder=3)
        ax.fill_between(decade["decade"], mean - sem, mean + sem, color=colour,
                        alpha=0.18, linewidth=0, zorder=2)
    ax.axvspan(1805, 1835, color=cfg.MUTED, alpha=0.07, linewidth=0, zorder=0)
    # Trend fits start here (FIT_FROM below).
    ax.axvline(1840, color=cfg.MUTED, linewidth=1, linestyle="--", zorder=1)
    ax.set_title(f"{letter}.  {panel}", fontsize=12, color=cfg.INK, loc="left",
                 pad=8)
    ax.set_xlabel("Decade")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.grid(True, color=cfg.GRID, linewidth=1, zorder=0)
    ax.set_axisbelow(True)
axes[0].set_ylabel("Mean per-book rate (per 1,000 sentences)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, fontsize=10.5, ncol=2,
           loc="lower center", bbox_to_anchor=(0.5, -0.06))
axes[0].text(1810, 8.4, "sparse\n(13-17 books\nper decade)", fontsize=8,
             color=cfg.MUTED, style="italic", va="bottom")
axes[0].text(1843, 8.4, "trend fit\nfrom 1840", fontsize=8,
             color=cfg.MUTED, style="italic", va="bottom")
fig.tight_layout()
cfg.save_fig(fig, "fig3_touch_sight_by_decade")

decade[["decade", "n_books"]].head(4)

## Trend slopes

In [ ]:
# Fitted from 1840: the 1810s-1830s rest on 13-17 books each and dominate any
# fit that includes them.
FIT_FROM = 1840
fit = decade[decade["decade"] >= FIT_FROM]

for label in cfg.TOUCH_SIGHT:
    slope = np.polyfit(fit["decade"] / 10, fit[f"{label}_mean"], 1)[0]
    first, last = fit[f"{label}_mean"].iloc[[0, -1]]
    print(f"  {label:22s} {first:5.1f} -> {last:5.1f}   {slope:+.2f} per decade")

## Chiasmus per book

In [ ]:
conform = ((books["rate_haptic_material"] > books["rate_haptic_human_animal"])
           & (books["rate_sight_human"] > books["rate_sight_material"]))
print(f"books showing the full chiasmus: {conform.mean():.0%}")

by_decade = (books.assign(conform=conform)[books["decade"] >= 1840]
             .groupby("decade")["conform"].agg(["mean", "size"]))
by_decade.assign(mean=lambda d: (d["mean"] * 100).round(0))

## Authors

In [ ]:
MIN_WORKS = 5


def clean_author(value):
    """'Last, First' entries -> 'First Last'; joint authorship kept as one id."""
    value = str(value).strip()
    if value in ("[]", "", "nan"):
        return None
    names = re.findall(r"'([^']*)'", value)
    if not names:
        return None
    out = []
    for name in names:
        if "," in name:
            last, first = name.split(",", 1)
            out.append(f"{first.strip()} {last.strip()}")
        else:
            out.append(name.strip())
    return " & ".join(out)


books["author_clean"] = books["author"].apply(clean_author)
authors = (books[books["author_clean"].notna()].groupby("author_clean")
           .agg(n_books=("filename", "count"),
                **{l: (f"rate_{l}", "mean") for l in cfg.TOUCH_SIGHT})
           .reset_index())
authors = authors[authors["n_books"] >= MIN_WORKS].reset_index(drop=True)
print(f"{len(authors)} authors with at least {MIN_WORKS} works")

## Dispersion

In [ ]:
rows = []
for label in cfg.TOUCH_SIGHT:
    v = authors[label]
    rows.append({"label": label, "median": v.median(),
                 "cv": v.std() / v.mean(), "skew": v.skew(),
                 "p90_p10": v.quantile(0.9) / v.quantile(0.1),
                 # share of authors within half to 1.5x the corpus median
                 "near_median": ((v >= 0.5 * v.median())
                                 & (v <= 1.5 * v.median())).mean()})
pd.DataFrame(rows).round(2)

## Figure 4

In [ ]:
ROWS = [("haptic_human_animal", "Touch \u2192 people"),
        ("haptic_material", "Touch \u2192 things"),
        ("sight_human", "Sight \u2192 people"),
        ("sight_material", "Sight \u2192 things")]

# One hue per marked author; every marked point is also labelled, so colour is
# a secondary cue. The paper's blue and green are avoided, since they carry the
# person/material distinction everywhere else.
MARKED = [("Jane Austen", "#7b2150"), ("Anthony Trollope", "#4a3aa7"),
          ("Charles Dickens", "#111111"), ("Rudyard Kipling", "#eda100"),
          ("Nathaniel Hawthorne", "#4d7c0f"), ("Herman Melville", "#c1440e"),
          ("Edgar Rice Burroughs", "#0f7b8a"),
          ("Anne Douglas Sedgwick", "#c2185b"), ("Henry James", "#8a5a2b"),
          ("George Eliot", "#4f6d7a")]


def surname(name):
    plain = re.sub(r"\s*\([^)]*\)", "", str(name)).strip().rstrip(".,")
    parts = [w for w in plain.split() if len(w.rstrip(".")) > 1]
    return parts[-1] if parts else plain


MIN_SEP = 0.085 * max(authors[l].max() for l, _ in ROWS)
rng = np.random.default_rng(0)

fig, ax = plt.subplots(figsize=(11, 8))
for i, (label, row_label) in enumerate(ROWS):
    y = len(ROWS) - 1 - i
    values = authors[label]
    ax.scatter(values, y + rng.uniform(-0.16, 0.16, len(values)), s=13,
               color=cfg.MUTED, alpha=0.28, edgecolors="none", zorder=2)
    ax.plot([values.median()] * 2, [y - 0.32, y + 0.32], color=cfg.SEC_INK,
            lw=1.5, zorder=3)

    # Labels alternate above and below the row, with a second tier when two
    # marked authors sit closer than MIN_SEP on the rate axis.
    points = sorted((values[authors["author_clean"] == name].iloc[0], name, c)
                    for name, c in MARKED
                    if (authors["author_clean"] == name).any())
    tiers = {True: [], False: []}
    for k, (x, name, colour) in enumerate(points):
        above = k % 2 == 0
        used = tiers[above]
        tier = 0
        while tier < len(used) and x - used[tier] < MIN_SEP:
            tier += 1
        if tier == len(used):
            used.append(x)
        else:
            used[tier] = x
        ax.scatter([x], [y], s=95, color=colour, edgecolors=cfg.SURFACE,
                   linewidths=1.3, zorder=5)
        ax.annotate(surname(name), (x, y),
                    xytext=(0, (15 + 13 * tier) * (1 if above else -1)),
                    textcoords="offset points", ha="center",
                    va="bottom" if above else "top", fontsize=8,
                    color=cfg.INK, zorder=6,
                    arrowprops=dict(arrowstyle="-", color=cfg.MUTED, lw=0.6,
                                    shrinkA=1.5, shrinkB=5))

ax.set_yticks(range(len(ROWS)))
ax.set_yticklabels([lab for _, lab in ROWS][::-1], fontsize=11, color=cfg.INK)
ax.set_ylim(-0.85, len(ROWS) - 0.05)
ax.set_xlabel("Mean per-book rate (per 1,000 sentences)")
ax.set_xlim(left=0)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.xaxis.grid(True, color=cfg.GRID, lw=0.8, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(left=False)
ax.legend([plt.Line2D([], [], color=cfg.SEC_INK, lw=1.5)], ["category median"],
          frameon=False, fontsize=9.5, loc="lower center",
          bbox_to_anchor=(0.5, -0.20), handletextpad=0.6)
fig.tight_layout()
cfg.save_fig(fig, "fig4_author_distribution")

print(f"{len(authors)} authors; rank of each marked author (1 = lowest)")
for name, _ in MARKED:
    ranks = [int(authors[l].rank(method="min")[
        authors["author_clean"] == name].iloc[0]) for l, _ in ROWS]
    print(f"  {surname(name):12s} " + "  ".join(f"{lab}: {r:3d}"
                                                for (_, lab), r in zip(ROWS, ranks)))